In [5]:
import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
from ytmusic_library import YTMusicPlaylists

import ytmusicapi as ytmusicapi
print(f'Using ytmusicapi version: {ytmusicapi.__version__}')

HEADER_FILE='../headers_auth.json'
print(f'Using header file: {HEADER_FILE}')

Y = YTMusicPlaylists(header=HEADER_FILE)
print(Y.playlists['title'].unique())

Using ytmusicapi version: 0.24.0
Using header file: ../headers_auth.json
['Your Likes' 'Acoustic Guitar Explorations' 'ambiant electro' 'ambient'
 'ambient BOC' 'ambient classic radio' 'ambient Dream Pop Deep Sleep'
 'ambient haunting harmonious' 'ambient Indie synths'
 'ambient Indie Synths radio' 'ambient lynchian radio'
 'ambient modern radio' 'ambient piano radio' 'beats' 'beats cosmic Slop'
 'Beats indie Chill' 'Beats indie Chill radio' 'beats instrumental'
 'Beats Lofi Loft' 'beats radio' 'beats Soulful Instrumentals'
 'beats trap dj' 'Beats Without Rhymes' "beats wonky LA '10 scene"
 'beats_chill dj' 'beats_jazzy dj' 'beats_lofi' 'beats_phat dj'
 'beats_raw dj' 'beats_soul dj' 'beats_wonky dj' 'bluegrass billy' 'blues'
 'blues chicago radio' 'blues delta radio' 'blues delta roots'
 'blues radio' 'blues texas roots' 'Bossa Nova' 'Bossa Nova radio'
 'Brass n chill' 'brass radio' 'Chill Supermix' 'Chillwave'
 'christmas crooners radio' 'electronic 2000s' 'electronic 2000s radio'
 '

# Process Each Playlist

##### TODO move based on playcount (if not LIKE infer NOT_LIKE based on large playcount)
##### TODO for NOT_LIKE?
##### TODO make into script, run monthly



In [6]:
# Skip playlistes inferred as these kind
DRY_RUN = False  # make sure all NOT OK is fine

VPRINT = False  # Verbose printing
WARN_PRINT = True  # Print warnings

# Options for checking inferred playlist kind
SKIP_PLAYLIST_KINDS = ('SKIP', 'ALBUM', 'YT_GENERATED')
LIKE_MIN_LIKE_PCT = 80
NOTLIKE_MAX_LIKE_PCT = 20
RADIO_MAX_LIKE_PCT = 50

# Options for processing playlists
MIN_RADIO_LIKE_TO_SPLIT = 10
DUPLICATE_THRESHOLD = 3  # was 4 first run


playlists_kinds = {k: set() for k in Y._valid_playlist_kinds}
for i, p in Y.playlists.iterrows():
    if VPRINT:
        print(100*'=' + f'\nPlaylist: {p.title} ({p.playlistId})',
              f'has {p.count} tracks')

    """Potentially skip playlist"""
    # Infer playlist kind from the title, default to LIKE if nothing inferred
    pl_kind = Y.infer_playlist_kind(p)
    if not pl_kind:
        pl_kind = 'LIKE'

    # Decide to skip playlist based on playlist kind
    playlists_kinds[pl_kind].add(p.title)
    if pl_kind in SKIP_PLAYLIST_KINDS:
        print(f'SKIPPING playlist: {p.title} as it is',
              f'a kind flagged for skipping: {pl_kind}')
        continue

    """Query playlist tracks then potentially skip"""
    # Query playlist tracks and other metadata
    p_info = Y.playlist_get_info(
        p.playlistId, playlist_limit=Y.playlist_limit).copy()

    # Check max length of playlist
    if len(p_info['tracks']) >= Y.playlist_limit:
        print(f'SKIPPING playlist: {p.title} which has',
              f'{Y.playlist_limit} or more tracks ({len(p_info["tracks"])})')
        continue

    # Check playlist privacy
    if p_info['privacy'] == 'PUBLIC':
        print(f'SKIPPING playlist: {p.title} which has',
              f'privacy: {p_info["privacy"]}')
        continue
    elif p_info['privacy'] == 'UNLISTED' and WARN_PRINT:
        print(f'WARNING {p_info["privacy"]} playlist: {p.title}')

    # Get ratings for playlist tracks
    ratings = {k: set() for k in Y._valid_ratings}
    for track in p_info["tracks"]:
        if track["likeStatus"] not in ratings.keys():
            ratings['NONE'].add(track["videoId"])
        else:
            ratings[track["likeStatus"]].add(track["videoId"])

    # See if playlist is correctly flagged as LIKE or RADIO
    like_percent = round(100*len(ratings["LIKE"])/len(p_info["tracks"]))
    if not Y._is_playlist_kind_ok(pl_kind, like_percent,
                                  LIKE_MIN_LIKE_PCT, NOTLIKE_MAX_LIKE_PCT,
                                  RADIO_MAX_LIKE_PCT):
        if WARN_PRINT:
            print(f'WARNING NOT OK {pl_kind} Playlist: {p.title}',
                  f'({like_percent}% liked) has: {len(ratings["LIKE"])} likes,',
                  f'{len(ratings["DISLIKE"])} dislikes,{len(ratings["INDIFFERENT"])}',
                  f'indifferent, {len(ratings["NONE"])} none')

    """Potentially alter playlist, or generate new playlists"""
    if DRY_RUN:
        continue

    # Remove duplicates from playlist
    new_pl_id = Y.playlist_remove_duplicates(
        p_info, duplicate_threshold=DUPLICATE_THRESHOLD, verbose=VPRINT)
    if p.playlistId != new_pl_id:
        p.playlistId = new_pl_id
        p_info = Y.playlist_get_info(
            new_pl_id, playlist_limit=Y.playlist_limit, use_cache=False)

    # Like all tracks in playlist if kind is LIKE
    if pl_kind == 'LIKE':
        Y.playlist_rate_all_songs(
            p_info, rating=pl_kind, skip_if_dislike=True, verbose=VPRINT)
        continue
    # Split radio playlist into LIKE vs RADIO
    elif pl_kind == 'INDIFFERENT':
        Y.move_likes_from_radio_playlist(
            p_info, min_n_like=MIN_RADIO_LIKE_TO_SPLIT, verbose=VPRINT)
        continue


SKIPPING playlist: Your Likes as it is a kind flagged for skipping: YT_GENERATED
Playlist Acoustic Guitar Explorations: Rated 0 of 56 tracks as LIKE
Playlist ambiant electro: Rated 0 of 65 tracks as LIKE
Playlist ambient: Rated 56 of 688 tracks as LIKE
Playlist ambient BOC: Rated 0 of 41 tracks as LIKE
Playlist ambient Dream Pop Deep Sleep: Rated 0 of 47 tracks as LIKE
Playlist ambient haunting harmonious: Rated 1 of 139 tracks as LIKE
Playlist ambient Indie synths: Rated 1 of 60 tracks as LIKE
Playlist beats: Rated 43 of 708 tracks as LIKE
Playlist beats cosmic Slop: Rated 0 of 35 tracks as LIKE
Playlist Beats indie Chill: Rated 0 of 56 tracks as LIKE
Playlist beats instrumental: Rated 27 of 266 tracks as LIKE
Playlist Beats Lofi Loft: Rated 0 of 63 tracks as LIKE
Playlist beats Soulful Instrumentals: Rated 11 of 213 tracks as LIKE
Playlist beats trap dj: Rated 5 of 69 tracks as LIKE
Playlist Beats Without Rhymes: Rated 1 of 45 tracks as LIKE
Playlist beats wonky LA '10 scene: Rated 2